# Medical Assistant - Part 2: Data Preparation

Simple notebook to prepare medical data for fine-tuning.

**Before starting:**
- Download datasets using `P1-DownloadDatasets.py`
- Datasets should be in `data/raw/` folder

## 1 - Setup

In [102]:
!pip install --quiet datasets==4.8.5 pandas==3.0.2


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [103]:
import json
import os
import pandas as pd
import random
import re
import shutil
from datasets import load_dataset
from pathlib import Path
from xml.etree import ElementTree as ET

# Clean up old data if it exists
if os.path.exists("data/processed"):
    shutil.rmtree("data/processed")
    print("Cleaned up old data/processed directory")

# Create fresh output folder
os.makedirs("data/processed", exist_ok=True)

# The cap was introduced to reduce the training time to something manageable for a free collab account
dataset_cap = 5000

print("✅ Setup complete")

Cleaned up old data/processed directory
✅ Setup complete


## 2 - Load PubMedQA

In [104]:
path = "data/raw/pubmedqa/data/ori_pqal.json"

with open(path, "r") as f:
    pubmedqa_raw = json.load(f)

pubmedqa_data = []

# Extract question and answer from each entry
for pubmed_id, item in pubmedqa_raw.items():
    question = item.get("QUESTION", "")
    answer = item.get("final_decision", "")
    long_answer = item.get("LONG_ANSWER", "")
    
    if question and answer:
        text = f"Question: {question}\nLong Answer: {long_answer}\nFinal Decision: {answer}"
        pubmedqa_data.append(text)

pubmedqa_data = pubmedqa_data[:dataset_cap]

print(f"Loaded {len(pubmedqa_data)} PubMedQA samples")
print(f"Example: {pubmedqa_data[0][:200]}...")

Loaded 1000 PubMedQA samples
Example: Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Long Answer: Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plan...


## 3 - Load MedQuAD

In [105]:
path = Path("data/raw/medquad")
medquad_data = []

# Walk through all topic folders (1_CancerGov_QA, 2_GARD_QA, etc.)
for topic_folder in sorted(path.glob("*_QA")):
    if not topic_folder.is_dir():
        continue
    
    # Walk through all XML files in the topic folder
    for xml_file in topic_folder.glob("*.xml"):
        try:
            tree = ET.parse(xml_file)
            root = tree.getroot()
            
            # MedQuAD has QAPairs that contain multiple QAPair elements
            qa_pairs = root.find("QAPairs")
            if qa_pairs is not None:
                for qa_pair in qa_pairs.findall("QAPair"):
                    question_elem = qa_pair.find("Question")
                    answer_elem = qa_pair.find("Answer")
                    
                    if question_elem is not None and answer_elem is not None:
                        question = question_elem.text
                        answer = answer_elem.text
                        
                        if question and answer:
                            text = f"Question: {question}\nAnswer: {answer}"
                            medquad_data.append(text)
        except:
            pass  # Skip problematic files

medquad_data = medquad_data[:dataset_cap]

print(f"Loaded {len(medquad_data)} MedQuAD samples")
print(f"Example: {medquad_data[0][:200]}...")

Loaded 5000 MedQuAD samples
Example: Question: What is (are) Non-Small Cell Lung Cancer ?
Answer: Key Points
                    - Non-small cell lung cancer is a disease in which malignant (cancer) cells form in the tissues of the lung....


## 4 - Load EPFL Guidelines

In [106]:
print("Downloading EPFL Guidelines...")
dataset = load_dataset("epfl-llm/guidelines")

epfl_data = []
for item in dataset["train"]:
    text = item.get("clean_text") or item.get("raw_text")
    if text:
        epfl_data.append(text)

epfl_data = epfl_data[:dataset_cap]

print(f"Loaded {len(epfl_data)} EPFL samples")
print(f"Example: {epfl_data[0][:200]}...")

Loaded 5000 EPFL samples
Example: # QUESTIONS Diagnosis/Staging
What benefit to clinical management does positron emission tomography (PET) or positron emission tomography/computed tomography (PET/CT) contribute to the diagnosis or st...


## 5 -  Load Synthea

In [107]:
synthea_encounters = []
synthea_conditions = []
synthea_medications = []
path = Path("data/raw/synthea/csv")

# Load encounters (clinical visits)
encounters_df = pd.read_csv(path / "encounters.csv")
for _, row in encounters_df.iterrows():
    description = row.get("DESCRIPTION", "")
    if description:
        synthea_encounters.append(f"Clinical encounter: {description}")

# Load conditions (diagnoses)
conditions_df = pd.read_csv(path / "conditions.csv")
for _, row in conditions_df.iterrows():
    description = row.get("DESCRIPTION", "")
    if description:
        synthea_conditions.append(f"Diagnosis: {description}")

# Load medications
medications_df = pd.read_csv(path / "medications.csv")
for _, row in medications_df.iterrows():
    description = row.get("DESCRIPTION", "")
    if description:
        synthea_medications.append(f"Medication: {description}")

synthea_data = synthea_encounters[:dataset_cap] + synthea_conditions[:dataset_cap] + synthea_medications[:dataset_cap]

print(f"Loaded {len(synthea_data)} Synthea samples")
print(f"Example: {synthea_data[0][:200]}...")

Loaded 15000 Synthea samples
Example: Clinical encounter: Encounter for symptom...


## 6 - Combine All Data

In [108]:
all_data = pubmedqa_data + medquad_data + epfl_data + synthea_data

print(f"Total samples: {len(all_data)}")
print(f"  - PubMedQA: {len(pubmedqa_data)}")
print(f"  - MedQuAD: {len(medquad_data)}")
print(f"  - EPFL: {len(epfl_data)}")
print(f"  - Synthea: {len(synthea_data)}")

Total samples: 26000
  - PubMedQA: 1000
  - MedQuAD: 5000
  - EPFL: 5000
  - Synthea: 15000


## 7 - Data clean up

In [109]:
cleaned_data = []
for text in all_data:
    # Normalize newlines and carriage returns to spaces
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    
    # Remove HTML/XML tags (if any slipped through)
    text = re.sub(r"<[^>]+>", "", text)
    
    # Remove extra whitespace
    text = " ".join(text.split())
    
    # Remove duplicate spaces
    text = re.sub(r"\s+", " ", text)
    
    # Replace dates with placeholder
    text = re.sub(r"\d{4}-\d{2}-\d{2}", "[DATE]", text)
    
    # Keep only non-empty samples (at least 20 characters)
    if len(text.strip()) > 20:
        cleaned_data.append(text)

print(f"After cleaning: {len(cleaned_data)} samples")
print(f"Filtered out: {len(all_data) - len(cleaned_data)} empty/short samples")

After cleaning: 25890 samples
Filtered out: 110 empty/short samples


## 8 - Save as JSONL

In [110]:
output_file = "data/processed/medical_data.jsonl"

with open(output_file, "w") as f:
    for text in cleaned_data:
        json.dump({"text": text}, f, ensure_ascii=False)
        f.write("\n")

print(f"✅ Saved {len(cleaned_data)} samples to {output_file}")

✅ Saved 25890 samples to data/processed/medical_data.jsonl


## 9 - Create Train/Validation Split

In [111]:
random.seed(42)
random.shuffle(cleaned_data)

split_point = int(len(cleaned_data) * 0.8)
train_data = cleaned_data[:split_point]
val_data = cleaned_data[split_point:]

# Save train
with open("data/processed/medical_data_train.jsonl", "w") as f:
    for text in train_data:
        json.dump({"text": text}, f, ensure_ascii=False)
        f.write("\n")

# Save validation
with open("data/processed/medical_data_val.jsonl", "w") as f:
    for text in val_data:
        json.dump({"text": text}, f, ensure_ascii=False)
        f.write("\n")

print(f"Training: {len(train_data)} samples (80%)")
print(f"Validation: {len(val_data)} samples (20%)")

Training: 20712 samples (80%)
Validation: 5178 samples (20%)


## 10 - View Statistics

In [112]:
word_counts = [len(text.split()) for text in cleaned_data]
char_counts = [len(text) for text in cleaned_data]

print(f"Average words per sample: {sum(word_counts) / len(word_counts):.0f}")
print(f"Average characters per sample: {sum(char_counts) / len(char_counts):.0f}")
print(f"\nMin words: {min(word_counts)}, Max words: {max(word_counts)}")
print(f"Min chars: {min(char_counts)}, Max chars: {max(char_counts)}")

Average words per sample: 1175
Average characters per sample: 7897

Min words: 2, Max words: 176664
Min chars: 21, Max chars: 1118313


## Done!

✅ Data preparation complete

Files created:
- `data/processed/medical_data_train.jsonl` - Training data
- `data/processed/medical_data_val.jsonl` - Validation data